# 01 tokenizer + model

目标：拆开 `AutoTokenizer` 和 `AutoModelForCausalLM`，看清楚文本如何变成 token ids，又如何 decode 回文本。


## 运行环境准备

这个 notebook 默认使用 `Qwen/Qwen2.5-0.5B-Instruct`，适合在魔搭 Notebook 里快速学习。

如果你想用更大的模型，可以把 `MODEL_ID` 改成 `Qwen/Qwen2.5-7B-Instruct`，然后重启内核重新运行。


In [1]:
from pathlib import Path

requirements_path = Path("requirements.txt")
if not requirements_path.exists():
    requirements_path = Path("../requirements.txt")

%pip install -r {requirements_path}



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
ERROR: Could not open requirements file: [Errno 2] No such file or directory: '../requirements.txt'
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from pathlib import Path

MODEL_ID = os.getenv("MODEL_ID", "Qwen/Qwen2.5-0.5B-Instruct")
MODEL_SOURCE = os.getenv("MODEL_SOURCE", "modelscope").lower()


def resolve_model_path(model_id):
    if Path(model_id).exists():
        return model_id
    if MODEL_SOURCE != "modelscope":
        return model_id

    from modelscope import snapshot_download
    return snapshot_download(model_id)


MODEL_PATH = resolve_model_path(MODEL_ID)
print("MODEL_ID =", MODEL_ID)
print("MODEL_PATH =", MODEL_PATH)


/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-05-19 22:31:23,345 - modelscope - INFO - Target directory already exists, skipping creation.


MODEL_ID = Qwen/Qwen2.5-0.5B-Instruct
MODEL_PATH = /mnt/workspace/.cache/modelscope/models/Qwen/Qwen2___5-0___5B-Instruct


## 1. 加载 tokenizer 和 model

这一层开始脱离 `pipeline`，显式控制模型和分词器。


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 553.47it/s]


## 2. messages 到 prompt

`chat_template` 负责把 system/user/assistant 消息渲染成模型训练时熟悉的格式。


In [4]:
messages = [
    {"role": "system", "content": "你是一个讲解大模型部署的老师。"},
    {"role": "user", "content": "用三句话解释什么是 chat template。"},
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

print(prompt)


<|im_start|>system
你是一个讲解大模型部署的老师。<|im_end|>
<|im_start|>user
用三句话解释什么是 chat template。<|im_end|>
<|im_start|>assistant



## 3. tokenize、generate、decode

`generate()` 输出包含输入 token 和新生成 token，所以 decode 时通常只取新增部分。


In [9]:
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
print(inputs)
outputs = model.generate(
    **inputs,
    max_new_tokens=120,
    do_sample=False,
)

new_token_ids = outputs[0][inputs["input_ids"].shape[-1] :]

print(new_token_ids)
answer = tokenizer.decode(new_token_ids, skip_special_tokens=True)

print(answer)


{'input_ids': tensor([[151644,   8948,    198,  56568, 101909, 105250,  26288, 104949, 102121,
           9370, 101049,   1773, 151645,    198, 151644,    872,    198,  11622,
          44991, 100908, 104136, 106582,   6236,   3811,   1773, 151645,    198,
         151644,  77091,    198]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1]], device='cuda:0')}
tensor([ 15672,  19911,  20412, 116976,  91282,  99692, 108421,   3837, 100751,
         50377, 105292, 105051,   1773, 100001, 108421, 102298,  98841,  29635,
          9370, 113940,   5373,  86119,  33108, 102104, 100166,   3837, 104193,
         20002,  73670, 104261,  29490,  57218,  72448,  71817, 108221,   1773,
         67338,  37029,   9686,  19911,   3837, 113129,  73670, 101098, 104004,
         20221, 100646, 109963, 105292,  99892,  57191,  47874,   3837,  68536,
        106431, 110867, 108598, 103991, 105051,   9370, 104449, 